In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Medical Search Model Analysis\n",
    "Analysis of model performance, embeddings, and search patterns"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "import os\n",
    "\n",
    "# Add project root to path\n",
    "project_root = Path.cwd().parent\n",
    "sys.path.append(str(project_root))\n",
    "\n",
    "# Import project modules\n",
    "from src.data.database import Database\n",
    "from config.settings import DB_CONFIG, MODEL_CONFIG\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from datetime import datetime, timezone\n",
    "import requests\n",
    "from sentence_transformers import SentenceTransformer\n",
    "\n",
    "%matplotlib inline\n",
    "plt.style.use('seaborn')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# SSL Certificate Setup\n",
    "def setup_ssl():\n",
    "    ssl_dir = project_root / 'ssl'\n",
    "    ssl_dir.mkdir(exist_ok=True)\n",
    "    cert_path = ssl_dir / 'rds-ca-bundle.pem'\n",
    "    timestamp_path = ssl_dir / 'cert_timestamp.txt'\n",
    "    \n",
    "    needs_update = True\n",
    "    if cert_path.exists() and timestamp_path.exists():\n",
    "        with open(timestamp_path, 'r') as f:\n",
    "            last_update = datetime.fromisoformat(f.read().strip())\n",
    "            if (datetime.now(timezone.utc) - last_update).days < 30:\n",
    "                needs_update = False\n",
    "    \n",
    "    if needs_update:\n",
    "        print(f\"Downloading SSL certificate at {datetime.now(timezone.utc).isoformat()}\")\n",
    "        response = requests.get('https://truststore.pki.rds.amazonaws.com/global/global-bundle.pem')\n",
    "        response.raise_for_status()\n",
    "        \n",
    "        with open(cert_path, 'wb') as f:\n",
    "            f.write(response.content)\n",
    "        \n",
    "        with open(timestamp_path, 'w') as f:\n",
    "            f.write(datetime.now(timezone.utc).isoformat())\n",
    "    \n",
    "    return str(cert_path)\n",
    "\n",
    "SSL_CERT_PATH = setup_ssl()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Initialize database connection\n",
    "db = Database()\n",
    "\n",
    "# Load model\n",
    "model = SentenceTransformer(MODEL_CONFIG['base_model'])\n",
    "\n",
    "def get_entity_embeddings(entity_type: str):\n",
    "    \"\"\"Get embeddings for specific entity type\"\"\"\n",
    "    query = f\"SELECT * FROM {entity_type} WHERE embedding IS NOT NULL\"\n",
    "    return pd.read_sql(query, db.engine)\n",
    "\n",
    "# Get embeddings for analysis\n",
    "doctor_embeddings = get_entity_embeddings('doctors')\n",
    "hospital_embeddings = get_entity_embeddings('hospitals')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Embedding Analysis\n",
    "Analyze distribution and clustering of embeddings"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "def analyze_embeddings(df, entity_type):\n",
    "    \"\"\"Analyze embeddings for an entity type\"\"\"\n",
    "    embeddings = np.stack(df['embedding'].apply(lambda x: np.frombuffer(x, dtype=np.float32)))\n",
    "    \n",
    "    # Dimensionality reduction for visualization\n",
    "    from sklearn.decomposition import PCA\n",
    "    pca = PCA(n_components=2)\n",
    "    embeddings_2d = pca.fit_transform(embeddings)\n",
    "    \n",
    "    # Plot\n",
    "    plt.figure(figsize=(10, 8))\n",
    "    plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], alpha=0.5)\n",
    "    plt.title(f'{entity_type} Embeddings Distribution')\n",
    "    plt.xlabel('PCA Component 1')\n",
    "    plt.ylabel('PCA Component 2')\n",
    "    plt.show()\n",
    "    \n",
    "    return embeddings_2d\n",
    "\n",
    "doctor_embeddings_2d = analyze_embeddings(doctor_embeddings, 'Doctor')\n",
    "hospital_embeddings_2d = analyze_embeddings(hospital_embeddings, 'Hospital')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Search Pattern Analysis\n",
    "Analyze common search patterns and success rates"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "source": [
    "# Get search patterns\n",
    "search_patterns = pd.read_sql(\"\"\"\n",
    "    SELECT \n",
    "        query,\n",
    "        search_type,\n",
    "        COUNT(*) as search_count,\n",
    "        AVG(results_count) as avg_results,\n",
    "        AVG(CASE WHEN results_count > 0 THEN 1 ELSE 0 END) as success_rate\n",
    "    FROM search_logs\n",
    "    GROUP BY query, search_type\n",
    "    HAVING COUNT(*) > 5\n",
    "    ORDER BY search_count DESC\n",
    "    LIMIT 20\n",
    "\"\"\", db.engine)\n",
    "\n",
    "# Plot search pattern analysis\n",
    "plt.figure(figsize=(15, 6))\n",
    "sns.barplot(data=search_patterns, x='query', y='success_rate')\n",
    "plt.xticks(rotation=45, ha='right')\n",
    "plt.title('Search Success Rates by Query')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 }
}